<a href="https://colab.research.google.com/github/saitejasrireddyveldi-collab/ML_Lab-2026-27/blob/main/Decision_Tree_Classfier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Introduction to decision**

A Decision Tree is a supervised machine learning algorithm used for both classification and regression tasks. It's one of the most intuitive and easy-to-understand models.

Think of it like a flowchart or a game of "20 Questions." The tree starts with a single root node representing the entire dataset. It then splits the data into smaller, more homogeneous groups based on a series of questions about the features. Each question corresponds to an internal node, and the possible answers are the branches. The process continues until it reaches the leaf nodes, which represent the final classification or prediction.

The goal during training is to find the best questions (splits) that most effectively separate the data into pure classes. To do this, it commonly uses metrics like the Gini Impurity or Entropy to measure the "disorder" or "impurity" of a node. A lower impurity means the data in that node is more uniform (i.e., belongs to a single class).

Key Advantages:

Easy to interpret: Its flowchart-like structure is simple to visualize and explain.

Handles both numerical and categorical data: It can work with different types of features.

Non-parametric: It doesn't make strong assumptions about the underlying distribution of the data.

Key Disadvantage:

Prone to overfitting: Without constraints (like limiting the tree's depth), it can create overly complex trees that memorize the training data but fail to generalize to new data. This is why hyperparameter tuning is crucial.

# **Scikit-learn (sklearn)**

Scikit-learn is a comprehensive, open-source machine learning library. It provides simple and efficient tools for data mining and data analysis.

sklearn.datasets: This module provides helper functions to load popular datasets, like the wine dataset (load_wine) used in the example. These built-in datasets are great for practicing and testing algorithms.

sklearn.model_selection: This module contains essential tools for splitting data and tuning models.

train_test_split: A function to randomly split a dataset into training and testing subsets. This is fundamental for evaluating a model's performance on unseen data.

GridSearchCV: An automated tool for hyperparameter tuning. It exhaustively searches through a specified grid of parameters and uses cross-validation to find the combination that yields the best performance.

sklearn.tree: This module contains the decision tree-based models.

DecisionTreeClassifier: The class that implements the Decision Tree algorithm for classification tasks.

sklearn.metrics: This module provides tools for evaluating model performance.

accuracy_score: A function that calculates the accuracy of a classification model, which is the proportion of correct predictions.

# **Basic Decision Tree Implementation**

First, let's build and train a simple Decision Tree classifier with its default settings.

In [1]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# Load the Adult Census Income dataset
adult = fetch_openml(
    name="adult",
    version=2,
    as_frame=True
)

X = adult.data
y = adult.target

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(
    include=["category", "object", "string"]
).columns

numerical_cols = X.select_dtypes(
    include=["number"]
).columns

# Preprocess categorical and numerical features
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        ),
        ("num", "passthrough", numerical_cols)
    ]
)

# Create the Decision Tree model
dt_default = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

# Train the model
dt_default.fit(X_train, y_train)

# Make predictions
y_pred_default = dt_default.predict(X_test)

# Evaluate the model
accuracy_default = accuracy_score(y_test, y_pred_default)

print(
    f"Accuracy of default Decision Tree: "
    f"{accuracy_default:.4f}"
)

Accuracy of default Decision Tree: 0.8187


# **2. Hyperparameter Tuning with GridSearchCV**

Hyperparameter tuning is the process of finding the optimal model architecture. For a Decision Tree, key parameters to tune include:

criterion: The function to measure the quality of a split ('gini' or 'entropy').

max_depth: The maximum depth of the tree. Limiting this helps prevent overfitting.

min_samples_split: The minimum number of samples required to split an internal node.

min_samples_leaf: The minimum number of samples required to be at a leaf node.

We will use GridSearchCV to test a "grid" of different parameter combinations and find the best one using cross-validation.

In [2]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# 1. Create a Pipeline with preprocessing and Decision Tree
dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

# 2. Define the parameter grid
param_grid = {
    "classifier__criterion": ["gini", "entropy"],
    "classifier__max_depth": [None, 5, 10, 15, 20],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4]
}

# 3. Set up GridSearchCV
grid_search = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    verbose=1,
    scoring="accuracy"
)

# 4. Fit GridSearchCV on training data
grid_search.fit(X_train, y_train)

# 5. Display the best parameters
print("\nBest Parameters found by GridSearchCV:")
print(grid_search.best_params_)

# Get the best model
best_dt = grid_search.best_estimator_

# 6. Evaluate the tuned model
y_pred_tuned = best_dt.predict(X_test)
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)

print(f"\nBest Cross-Validation Accuracy: "
      f"{grid_search.best_score_:.4f}")

print(f"Accuracy of default Decision Tree: "
      f"{accuracy_default:.4f}")

print(f"Accuracy of tuned Decision Tree: "
      f"{accuracy_tuned:.4f}")

Fitting 5 folds for each of 90 candidates, totalling 450 fits

Best Parameters found by GridSearchCV:
{'classifier__criterion': 'gini', 'classifier__max_depth': 10, 'classifier__min_samples_leaf': 4, 'classifier__min_samples_split': 2}

Best Cross-Validation Accuracy: 0.8567
Accuracy of default Decision Tree: 0.8187
Accuracy of tuned Decision Tree: 0.8606


# **Explanation of the Tuning Process**

Define Parameter Grid: We create a dictionary (param_grid) where keys are the parameter names (max_depth, etc.) and values are lists of settings to try for those parameters.

Instantiate GridSearchCV: We pass our base model (dt), the parameter grid, and a cross-validation strategy (cv=5). GridSearchCV will now train and evaluate the model for every single combination of parameters in the grid. For example, it will test a tree with criterion='gini', max_depth=5, min_samples_split=2, min_samples_leaf=1, and so on.

Fit: Calling .fit() starts this exhaustive search process on the training data.

Best Estimator: After the search is complete, grid_search.best_params_ gives you the combination that performed the best, and grid_search.best_estimator_ gives you the model already trained with these optimal parameters.

Final Evaluation: By comparing the accuracy of the default model and the tuned model on the test set, you can see the improvement gained from hyperparameter tuning. Typically, the tuned model has better (or at least more reliable) performance.